> **Registration unlock run (new):** for the CPU + GPU registration-unlock job chain (GPU dense pose scan, joint registration, pose verifier, retrained pair classifier, Cellpose-SAM self-training, `submission_v13_*.csv` candidates and `hpc_unlock_report.md`), open **`CellMatch_HPC_Unlock.ipynb`** in this folder instead and follow its numbered cells (Steps 0–5).
>
> This notebook is unchanged: its Run-All cell submits the older training chain and overwrites `submission.csv`. Do not run both notebooks for the same submission. `submission_v10_cpgate.csv` stays the safe final either way.

# CellMatch on NYU Cloud Bursting — Run All → `submission.csv`

**Run > Run All Cells.**

Works in two modes:
1. **GPU Jupyter** (best): runs training in this session.
2. **Any Jupyter / login shell** (this CPU session is fine): cell submits Slurm jobs on `g2-standard-12` (L4) or `c12m85-a100-1` (A100) and waits until `submission.csv` appears.

Account: `cs_gy_6923-2026fa`. Project: `/scratch/ts5789/cellmatch`.


In [ ]:
from pathlib import Path
import os

WORK = Path("/scratch/ts5789/cellmatch")
assert WORK.exists(), f"Missing {WORK} — clone Cell-Matching there first"
os.chdir(WORK)
os.environ["SLURM_SUBMIT_DIR"] = str(WORK)
os.environ["WORK_DIR"] = str(WORK)
os.environ["CELLPOSE_LOCAL_MODELS_PATH"] = str(WORK / "cellpose_weights")
os.environ["PIP_CACHE_DIR"] = str(WORK / ".pip_cache")
for p in ["logs", "cellpose_weights", ".pip_cache", "models", "cache"]:
    (WORK / p).mkdir(exist_ok=True)
print("WORK =", WORK.resolve())
ACCOUNT = "cs_gy_6923-2026fa"
# Change to c12m85-a100-1 if you want A100 instead of L4
GPU_PARTITION = "g2-standard-12"
GPU_MEM = "40G"
CPU_PARTITION = "n2c48m24"
CPU_MEM = "20G"


In [ ]:
from pathlib import Path
WORK = Path("/scratch/ts5789/cellmatch")
DATA = WORK / "Project_2_Dataset"
assert (DATA / "training/train_ground_truth.csv").exists(), "dataset missing under Project_2_Dataset"
print("training:", sorted(p.name for p in (DATA/"training").glob("subject_*")))
print("test regions:", len(list((DATA/"hidden_test").glob("*/*"))))
print("env present:", (WORK/"env"/"bin"/"python").exists())


In [ ]:
# Run All: if this session has a GPU, train here; else submit setup→gpu→cpu and wait.
import os, subprocess, time
from pathlib import Path

WORK = Path("/scratch/ts5789/cellmatch")
PY = WORK / "env" / "bin" / "python"
SUB = WORK / "submission.csv"
ACCOUNT = "cs_gy_6923-2026fa"
GPU_PARTITION = "g2-standard-12"   # or "c12m85-a100-1" for A100
GPU_MEM = "40G"
CPU_PARTITION = "n2c48m24"
CPU_MEM = "20G"

def cuda_ok():
    if not PY.exists():
        return False
    try:
        out = subprocess.check_output(
            [str(PY), "-c", "import torch; print(torch.cuda.is_available())"], text=True).strip()
        return out == "True"
    except Exception:
        return False

def sbatch(extra, script):
    env = {k: v for k, v in os.environ.items() if not k.startswith("SLURM_")}
    env["PATH"] = os.environ.get("PATH", "/usr/bin:/bin:/opt/slurm/bin")
    cmd = ["/opt/slurm/bin/sbatch", "--parsable", "-A", ACCOUNT, "--export=NONE", *extra, script]
    print("+", " ".join(cmd), flush=True)
    return subprocess.check_output(cmd, text=True, cwd=str(WORK), env=env).strip()

def wait_job(jobid, label):
    while True:
        state = subprocess.check_output(
            ["squeue", "-h", "-j", jobid, "-o", "%T"], text=True, stderr=subprocess.DEVNULL).strip()
        if not state:
            acct = subprocess.check_output(
                ["sacct", "-j", jobid, "-n", "-X", "-o", "State"], text=True).strip().split()[0]
            print(label, jobid, "->", acct, flush=True)
            if acct.startswith("COMPLETED"):
                return
            raise RuntimeError(f"{label} job {jobid} ended as {acct}. Check logs/")
        print(label, jobid, state, time.strftime("%H:%M:%S"), flush=True)
        time.sleep(60)

if cuda_ok():
    print("GPU in this session — running pipeline locally")
    subprocess.check_call([str(PY), "run_hpc.py", "gpu"], cwd=str(WORK))
    subprocess.check_call([str(PY), "run_hpc.py", "cpu"], cwd=str(WORK))
else:
    print("No GPU here — submitting Slurm chain")
    print(f"GPU_PARTITION={GPU_PARTITION} (edit this cell for A100)")
    deps = []
    if not PY.exists():
        setup = sbatch(["-p", CPU_PARTITION, "--mem=16G"], "hpc/setup.sbatch")
        deps = [f"--dependency=afterok:{setup}"]
        print("setup", setup)
    gpu = sbatch(["-p", GPU_PARTITION, f"--mem={GPU_MEM}", "--gres=gpu:1", *deps], "hpc/gpu.sbatch")
    cpu = sbatch(["-p", CPU_PARTITION, f"--mem={CPU_MEM}", f"--dependency=afterok:{gpu}"], "hpc/cpu.sbatch")
    print("gpu", gpu, "cpu", cpu)
    if deps:
        wait_job(setup, "setup")
    wait_job(gpu, "gpu")
    wait_job(cpu, "cpu")

assert SUB.exists(), "submission.csv missing"
subprocess.check_call([str(PY), "validate_submission.py", str(SUB)], cwd=str(WORK))
print("DONE →", SUB, f"({SUB.stat().st_size:,} bytes)")
print("\n".join(SUB.read_text().splitlines()[:3]))
